# CLF Correction with 1D UNet

This notebook integrates:
1. 1D UNet model for trajectory generation
2. CLF (Control Lyapunov Function) correction
3. Visualization and analysis

In [9]:
# Import necessary libraries
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
import torchdiffeq
from scipy.optimize import minimize
import os
import time
from datetime import datetime

# fmtorch imports
from fmtorch.scheduler import CondOTScheduler
from fmtorch.path import AffinePath
from fmtorch.solver import ODESolver
from fmtorch.utils import ModelWrapper

print("All libraries imported successfully")

All libraries imported successfully


## 定义1D UNet模型

In [10]:
class Mish(nn.Module):
    def forward(self, x):
        return x * torch.tanh(F.softplus(x))


class SinusoidalPosEmb(nn.Module):
    """时间步编码 - 修复版，支持标量和批次输入"""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        
        # 处理不同维度的时间输入
        if t.dim() == 0:  # 标量
            t = t.unsqueeze(0)
        if t.dim() == 1:  # (batch,)
            t = t.unsqueeze(-1)  # (batch, 1)
        
        emb = t * emb[None, :]  # (batch, half_dim)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)  # (batch, dim)
        return emb


class ResBlock1D(nn.Module):
    """简化的1D残差块"""
    def __init__(self, channels, time_emb_dim):
        super().__init__()
        
        self.time_mlp = nn.Sequential(
            Mish(),
            nn.Linear(time_emb_dim, channels)
        )
        
        self.block = nn.Sequential(
            nn.GroupNorm(8, channels),
            Mish(),
            nn.Conv1d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            Mish(),
            nn.Conv1d(channels, channels, 3, padding=1),
        )
    
    def forward(self, x, t_emb):
        h = self.block(x)
        # 确保 t_emb 和 x 的 batch 维度匹配
        if t_emb.shape[0] == 1 and x.shape[0] > 1:
            t_emb = t_emb.expand(x.shape[0], -1)
        h = h + self.time_mlp(t_emb)[:, :, None]
        return x + h


class SimpleUNet1D(nn.Module):
    """
    简化的1D UNet
    Input: (batch, 2, 100)
    Output: (batch, 2, 100)
    """
    def __init__(self, time_emb_dim=128, hidden_dim=128):
        super().__init__()
        
        self.time_emb_dim = time_emb_dim
        self.hidden_dim = hidden_dim
        
        # 时间嵌入
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            Mish(),
            nn.Linear(time_emb_dim * 4, time_emb_dim)
        )
        
        # 编码器
        self.conv_in = nn.Conv1d(2, hidden_dim, 3, padding=1)
        
        # Level 1: 100 -> 50
        self.down1_block = nn.ModuleList([
            ResBlock1D(hidden_dim, time_emb_dim),
            ResBlock1D(hidden_dim, time_emb_dim)
        ])
        self.down1_sample = nn.Conv1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        
        # Level 2: 50 -> 25
        self.down2_block = nn.ModuleList([
            ResBlock1D(hidden_dim, time_emb_dim),
            ResBlock1D(hidden_dim, time_emb_dim)
        ])
        self.down2_sample = nn.Conv1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        
        # 中间层: 25
        self.mid_block = nn.ModuleList([
            ResBlock1D(hidden_dim, time_emb_dim),
            ResBlock1D(hidden_dim, time_emb_dim)
        ])
        
        # 解码器
        # Level 2: 25 -> 50
        self.up2_sample = nn.ConvTranspose1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        self.up2_block = nn.ModuleList([
            ResBlock1D(hidden_dim * 2, time_emb_dim),
            ResBlock1D(hidden_dim * 2, time_emb_dim)
        ])
        self.up2_conv = nn.Conv1d(hidden_dim * 2, hidden_dim, 1)
        
        # Level 1: 50 -> 100
        self.up1_sample = nn.ConvTranspose1d(hidden_dim, hidden_dim, 4, stride=2, padding=1)
        self.up1_block = nn.ModuleList([
            ResBlock1D(hidden_dim * 2, time_emb_dim),
            ResBlock1D(hidden_dim * 2, time_emb_dim)
        ])
        self.up1_conv = nn.Conv1d(hidden_dim * 2, hidden_dim, 1)
        
        # 输出层
        self.out_block = nn.ModuleList([
            ResBlock1D(hidden_dim, time_emb_dim),
            ResBlock1D(hidden_dim, time_emb_dim)
        ])
        self.conv_out = nn.Conv1d(hidden_dim, 2, 3, padding=1)
    
    def forward(self, x, t):
        """
        Args:
            x: (batch, 2, 100)
            t: (batch,) or scalar
        Returns:
            (batch, 2, 100)
        """
        # 时间嵌入
        t_emb = self.time_mlp(t)
        
        # 编码器
        h = self.conv_in(x)  # (batch, hidden_dim, 100)
        
        # Down 1
        for block in self.down1_block:
            h = block(h, t_emb)
        h1 = h  # Skip connection
        h = self.down1_sample(h)  # (batch, hidden_dim, 50)
        
        # Down 2
        for block in self.down2_block:
            h = block(h, t_emb)
        h2 = h  # Skip connection
        h = self.down2_sample(h)  # (batch, hidden_dim, 25)
        
        # 中间
        for block in self.mid_block:
            h = block(h, t_emb)
        
        # Up 2
        h = self.up2_sample(h)  # (batch, hidden_dim, 50)
        h = torch.cat([h, h2], dim=1)  # (batch, hidden_dim*2, 50)
        for block in self.up2_block:
            h = block(h, t_emb)
        h = self.up2_conv(h)  # (batch, hidden_dim, 50)
        
        # Up 1
        h = self.up1_sample(h)  # (batch, hidden_dim, 100)
        h = torch.cat([h, h1], dim=1)  # (batch, hidden_dim*2, 100)
        for block in self.up1_block:
            h = block(h, t_emb)
        h = self.up1_conv(h)  # (batch, hidden_dim, 100)
        
        # 输出
        for block in self.out_block:
            h = block(h, t_emb)
        out = self.conv_out(h)  # (batch, 2, 100)
        
        return out

print("1D UNet model defined")

1D UNet model defined


## 加载训练好的模型

In [11]:
# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 创建模型
unet_1d = SimpleUNet1D(time_emb_dim=128, hidden_dim=128).to(device)

# 加载训练好的权重
# TODO: 修改为你的模型权重文件路径
model_path = 'path_to_your_trained_model.pth'  # ← 需要修改

if os.path.exists(model_path):
    unet_1d.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded from {model_path}")
else:
    print(f"Warning: Model file not found at {model_path}")
    print("Please train the model first or provide the correct path")

unet_1d.eval()

# 包装为Flow Matching模型
fm_model = ModelWrapper(unet_1d)
print("Model wrapped for Flow Matching")

Using device: cuda
Please train the model first or provide the correct path
Model wrapped for Flow Matching


## 车辆动力学仿真函数

In [12]:
def simulate_vehicle(x0, u_seq, dt=0.1):
    """
    Simulate vehicle dynamics
    
    Args:
        x0: initial state [x, y, theta, v, r]
        u_seq: control sequence, shape (control_steps, 2), each step: [a, delta]
        dt: time step
    
    Returns:
        states: state sequence, shape (state_steps, 5)
    """
    state_steps = len(u_seq) + 1
    states = np.zeros((state_steps, 5))
    states[0] = x0
    
    for k in range(len(u_seq)):
        x, y, theta, v, r = states[k]
        a, delta = u_seq[k]
        
        # Euler integration
        x_next = x + dt * v * np.cos(theta)
        y_next = y + dt * v * np.sin(theta)
        theta_next = theta + dt * r
        v_next = v + dt * a
        r_next = r + dt * delta
        
        states[k+1] = [x_next, y_next, theta_next, v_next, r_next]
    
    return states

print("Vehicle dynamics function defined")

Vehicle dynamics function defined


## 数据归一化类

In [13]:
class Scaler:
    def __init__(self, data_min, data_max):
        self.min = data_min
        self.max = data_max
    
    def normalize(self, x):
        return 2 * (x - self.min) / (self.max - self.min) - 1
    
    def denormalize(self, x_norm):
        return (x_norm + 1) / 2 * (self.max - self.min) + self.min

print("Scaler class defined")

Scaler class defined


## CLF修正相关函数

In [14]:
def time_blow_up_function(t, T=1.0, cap=1e3):
    """
    Time blow-up function: when t→T, phi → ∞, but capped at cap
    """
    phi = 1.0 / (T - t + 1e-8)
    return min(phi, cap)

def correct_with_clf(states_phys, controls_phys, clf_params):
    """
    Correct control sequence using CLF
    
    Args:
        states_phys: physical state sequence (100,3) [x, y, theta]
        controls_phys: physical control sequence (100,2) [a, delta]
        clf_params: CLF parameters dictionary
    
    Returns:
        corrected control sequence
    """
    N = states_phys.shape[0]
    a_max = clf_params['a_max']
    d_max = clf_params['delta_max']
    λ_reg = clf_params['lambda_reg']
    β = clf_params['beta']
    mask = clf_params.get('mask', 0.0)
    t = clf_params.get('t_frac', 1.0)
    cap = clf_params.get('cap', 1e3)
    dt = clf_params.get('dt', 0.1)

    # Skip correction if t_frac < mask
    if t < mask:
        return controls_phys.copy()

    # Calculate blow_up weight
    phi = time_blow_up_function(t, T=1.0, cap=cap)
    w_state = phi / (1.0 + phi)  # ∈ (0,1)

    # Initial physical state
    x0 = np.array([states_phys[0,0],
                   states_phys[0,1],
                   states_phys[0,2],
                   0.0, 0.0])

    def objective(u_flat):
        u_seq = u_flat.reshape((N, 2))
        states_sim = simulate_vehicle(x0, u_seq, dt=dt)
        
        # State deviation
        pos_dev = 0.0
        for k in range(N):
            dx = states_sim[k, 0] - states_phys[k, 0]
            dy = states_sim[k, 1] - states_phys[k, 1]
            pos_dev += dx**2 + dy**2
        
        # Control smoothness
        smooth_u = 0.0
        for k in range(N-1):
            du = u_seq[k+1] - u_seq[k]
            smooth_u += np.sum(du**2)
        
        loss = w_state * pos_dev + β * smooth_u
        return loss

    # Constraints
    bounds = [(-a_max, a_max), (-d_max, d_max)] * N
    u0_flat = controls_phys.flatten()
    
    result = minimize(
        objective,
        u0_flat,
        method='L-BFGS-B',
        bounds=bounds,
        options={'maxiter': 100, 'ftol': 1e-6}
    )
    
    u_corrected = result.x.reshape((N, 2))
    return u_corrected

print("CLF correction functions defined")

CLF correction functions defined


## 使用Flow Matching生成轨迹

In [15]:
def generate_trajectory_with_flow_matching(num_points=100):
    """
    使用1D UNet和Flow Matching生成轨迹
    
    Returns:
        states: (100, 3) - [x, y, theta]
        controls: (100, 2) - [a, delta] 
        trajectory: (2, 100) - [x, y] for visualization
    """
    solver = ODESolver(velocity_model=fm_model)
    
    with torch.no_grad():
        # 生成初始噪声 (1, 2, 100)
        noise_traj = torch.randn((1, 2, num_points), device=device)
        
        # Flow matching生成
        time_grid = torch.tensor([0.0, 1.0], device=device)
        generated_traj = solver.sample(
            x_init=noise_traj,
            time_grid=time_grid,
            method='dopri5',
            atol=1e-4,
            rtol=1e-4,
            return_intermediates=False
        )
        
        # 转换为numpy (2, 100)
        trajectory = generated_traj[0].cpu().numpy()
    
    # 从XY轨迹计算theta
    x = trajectory[0]  # (100,)
    y = trajectory[1]  # (100,)
    
    # 计算方向角
    theta = np.zeros(num_points)
    for i in range(num_points - 1):
        dx = x[i+1] - x[i]
        dy = y[i+1] - y[i]
        theta[i] = np.arctan2(dy, dx)
    theta[-1] = theta[-2]  # 最后一个点使用前一个的角度
    
    # 组合状态 (100, 3)
    states = np.column_stack([x, y, theta])
    
    # 生成简单的控制序列（可以改进）
    controls = np.zeros((num_points, 2))
    for i in range(num_points - 1):
        # 简单的差分近似
        dtheta = theta[i+1] - theta[i]
        controls[i] = [0.1, dtheta]  # [a, delta]
    
    return states, controls, trajectory

print("Trajectory generation function defined")

Trajectory generation function defined


## 生成并修正单个轨迹（测试）

In [17]:
# 生成轨迹
print("Generating trajectory with Flow Matching...")
states_orig, controls_orig, traj_xy = generate_trajectory_with_flow_matching()

# CLF参数
clf_params = {
    'a_max': 2.0,
    'delta_max': 0.5,
    'lambda_reg': 0.1,
    'beta': 1.0,
    'mask': 0.0,
    't_frac': 1.0,
    'cap': 1e3,
    'dt': 0.1
}

# 应用CLF修正
print("Applying CLF correction...")
controls_corrected = correct_with_clf(states_orig, controls_orig, clf_params)

# 用修正后的控制重新仿真
x0 = np.array([states_orig[0,0], states_orig[0,1], states_orig[0,2], 0.0, 0.0])
states_corrected = simulate_vehicle(x0, controls_corrected)

print("Done!")
print(f"Original states shape: {states_orig.shape}")
print(f"Corrected states shape: {states_corrected.shape}")
print(f"Original controls shape: {controls_orig.shape}")
print(f"Corrected controls shape: {controls_corrected.shape}")

Generating trajectory with Flow Matching...


TypeError: ODESolver.sample() missing 1 required positional argument: 'step_size'

## 可视化对比

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# (a) XY Trajectory
axs[0, 0].plot(states_orig[:, 0], states_orig[:, 1], 'b-', lw=2, label='Original')
axs[0, 0].plot(states_corrected[:, 0], states_corrected[:, 1], 'r--', lw=2, label='CLF-Corrected')
axs[0, 0].set_xlabel('x [m]')
axs[0, 0].set_ylabel('y [m]')
axs[0, 0].set_title('Trajectory Comparison')
axs[0, 0].axis('equal')
axs[0, 0].legend()
axs[0, 0].grid(True, alpha=.3)

# (b) Position Error
min_len = min(len(states_corrected), len(states_orig))
pos_dev = np.linalg.norm(states_corrected[:min_len, :2] - states_orig[:min_len, :2], axis=1)
t_axis = np.arange(min_len) * 0.1
axs[0, 1].plot(t_axis, pos_dev, 'k-')
axs[0, 1].set_xlabel('t [s]')
axs[0, 1].set_ylabel('||Δp|| [m]')
axs[0, 1].set_title('Position Deviation')
axs[0, 1].grid(True, alpha=.3)

# (c) Control Acceleration
min_len_ctrl = min(len(controls_orig), len(controls_corrected))
t_axis_ctrl = np.arange(min_len_ctrl) * 0.1
axs[1, 0].plot(t_axis_ctrl, controls_orig[:min_len_ctrl, 0], 'b-', label='Original a')
axs[1, 0].plot(t_axis_ctrl, controls_corrected[:min_len_ctrl, 0], 'r--', label='Corrected a')
axs[1, 0].set_xlabel('t [s]')
axs[1, 0].set_ylabel('accel [m/s²]')
axs[1, 0].legend()
axs[1, 0].grid(True, alpha=.3)

# (d) Control Steering
axs[1, 1].plot(t_axis_ctrl, controls_orig[:min_len_ctrl, 1], 'b-', label='Original δ')
axs[1, 1].plot(t_axis_ctrl, controls_corrected[:min_len_ctrl, 1], 'r--', label='Corrected δ')
axs[1, 1].set_xlabel('t [s]')
axs[1, 1].set_ylabel('steer [rad]')
axs[1, 1].legend()
axs[1, 1].grid(True, alpha=.3)

fig.suptitle("CLF Correction with 1D UNet: Before/After Comparison", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 批量生成和修正（可选）

In [ ]:
# 批量生成修正后的轨迹
n_samples = 10  # 可以修改数量
batch_results = {
    'original_states': [],
    'original_controls': [],
    'corrected_states': [],
    'corrected_controls': []
}

print(f"Generating {n_samples} samples with CLF correction...")
for i in range(n_samples):
    print(f"Sample {i+1}/{n_samples}...", end=' ')
    
    # 生成
    states_orig, controls_orig, _ = generate_trajectory_with_flow_matching()
    
    # 修正
    controls_corr = correct_with_clf(states_orig, controls_orig, clf_params)
    x0 = np.array([states_orig[0,0], states_orig[0,1], states_orig[0,2], 0.0, 0.0])
    states_corr = simulate_vehicle(x0, controls_corr)
    
    # 保存
    batch_results['original_states'].append(states_orig)
    batch_results['original_controls'].append(controls_orig)
    batch_results['corrected_states'].append(states_corr[:, :3])  # 只保留x,y,theta
    batch_results['corrected_controls'].append(controls_corr)
    
    print("Done")

print(f"\nAll {n_samples} samples generated successfully!")

# 可选：保存结果
import pickle
with open('batch_results_1dunet.pkl', 'wb') as f:
    pickle.dump(batch_results, f)
print("Results saved to 'batch_results_1dunet.pkl'")